# 🍎 Apple Disease Classification — CNN (Fixed for train/test folder structure)
### Dataset Structure:
```
apple_disease_classification/
├── train/
│   ├── class_1/
│   ├── class_2/
│   └── ...
└── test/
    ├── class_1/
    ├── class_2/
    └── ...
```
### Fixes Applied:
- ✅ `flow_from_directory` used (matches your folder structure)
- ✅ Validation split taken from train folder (10%)
- ✅ Data Augmentation on training only
- ✅ BatchNormalization after every Conv block
- ✅ GlobalAveragePooling2D (less overfitting than Flatten)
- ✅ L2 Regularization on Dense layers
- ✅ EarlyStopping + ReduceLROnPlateau + ModelCheckpoint
- ✅ Class weights for imbalanced data
- ✅ 4 Conv blocks (enough capacity, no underfitting)

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Imports

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

## Step 3: Extract Dataset

In [ ]:
zip_path   = '/content/drive/MyDrive/apple_disease_classification.zip'
extract_to = '/content/apple_disease_classification'

if not os.path.exists(extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print(f"Extraction complete: {extract_to}")
else:
    print(f"Already extracted: {extract_to}")

TRAIN_DIR = os.path.join(extract_to, 'train')
TEST_DIR  = os.path.join(extract_to, 'test')

print(f"\nTrain folder: {TRAIN_DIR}")
print(f"Test  folder: {TEST_DIR}")
print(f"Classes found in train: {sorted(os.listdir(TRAIN_DIR))}")

## Step 4: Class Distribution Check

In [ ]:
class_names = sorted(os.listdir(TRAIN_DIR))
class_counts = {cls: len(os.listdir(os.path.join(TRAIN_DIR, cls))) for cls in class_names}

print("--- Train Class Distribution ---")
for cls, count in class_counts.items():
    print(f"  {cls:40s}: {count} images")

counts = list(class_counts.values())
imbalance_ratio = max(counts) / min(counts)
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}")
if imbalance_ratio > 1.5:
    print("  Imbalanced — class_weight will be applied.")
else:
    print(" Balanced")

plt.figure(figsize=(14, 5))
plt.bar(class_counts.keys(), class_counts.values(), color='steelblue', edgecolor='black')
plt.title('Train Class Distribution', fontsize=14)
plt.ylabel('Number of Images')
plt.xlabel('Class')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 5: Data Generators
> **Training** → Augmentation + Normalization (fights overfitting)  
> **Validation** → Normalization only (taken as 10% split from train)  
> **Test** → Normalization only, shuffle=False (for correct evaluation)

In [ ]:
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32

# ── Training generator: augmentation + 10% validation split ────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.1    # 10% of train → validation
)

# ── Test generator: only normalization ─────────────────────────────────────
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False    # IMPORTANT: must be False for correct y_true
)

NUM_CLASSES = len(train_gen.class_indices)
print(f"\nNumber of classes : {NUM_CLASSES}")
print(f"Train samples     : {train_gen.samples}")
print(f"Val   samples     : {val_gen.samples}")
print(f"Test  samples     : {test_gen.samples}")

## Step 6: Compute Class Weights (handles imbalanced classes)

In [ ]:
# Get integer labels for all training samples
train_labels = train_gen.classes

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weight_dict = dict(enumerate(class_weights_array))
print("Class weights:")
for idx, weight in class_weight_dict.items():
    cls_name = [k for k, v in train_gen.class_indices.items() if v == idx][0]
    print(f"  [{idx}] {cls_name:40s} → weight: {weight:.3f}")

## Step 7: CNN Model Architecture

| Layer Block | Purpose |
|---|---|
| Conv2D × 2 + BN + Pool + Dropout | Feature extraction, stable training |
| 4 blocks (32→64→128→256 filters) | Enough capacity → no underfitting |
| GlobalAveragePooling2D | Fewer params than Flatten → less overfitting |
| Dense + L2 + BN + Dropout(0.4) | Strong regularization in head |

In [ ]:
def build_model(num_classes, input_shape=(150, 150, 3)):
    model = Sequential([

        # ── Block 1: 32 filters ──────────────────────────────────
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        Conv2D(32, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Dropout(0.25),

        # ── Block 2: 64 filters ──────────────────────────────────
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Dropout(0.25),

        # ── Block 3: 128 filters ─────────────────────────────────
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Dropout(0.25),

        # ── Block 4: 256 filters ─────────────────────────────────
        Conv2D(256, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Dropout(0.3),

        # ── Classification Head ───────────────────────────────────
        GlobalAveragePooling2D(),          # replaces Flatten — fewer params
        Dense(256, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(NUM_CLASSES)
model.summary()

## Step 8: Callbacks

In [ ]:
callbacks = [
    # Stop training when val_loss stops improving
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate when stuck
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    # Save best model automatically
    ModelCheckpoint(
        filepath='best_apple_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print(" Callbacks ready.")

## Step 9: Train the Model

In [ ]:
EPOCHS = 30   # EarlyStopping will stop early if overfitting detected

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

## Step 10: Training Curves (Overfitting / Underfitting Check)

In [ ]:
acc      = history.history['accuracy']
val_acc  = history.history['val_accuracy']
loss     = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc,     label='Train Accuracy',      linewidth=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', linewidth=2, linestyle='--')
plt.title('Accuracy Curve', fontsize=13)
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss,     label='Train Loss',      linewidth=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', linewidth=2, linestyle='--')
plt.title('Loss Curve', fontsize=13)
plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.legend(); plt.grid(alpha=0.3)

plt.suptitle('Training vs Validation — Fit Quality Check', fontsize=14)
plt.tight_layout()
plt.show()

# ── Diagnosis ───────────────────────────────────────────────────────────────
final_gap = abs(acc[-1] - val_acc[-1])
print(f"\nTrain Accuracy      : {acc[-1]*100:.2f}%")
print(f"Validation Accuracy : {val_acc[-1]*100:.2f}%")
print(f"Accuracy Gap        : {final_gap*100:.2f}%")

if final_gap > 0.10:
    print("  Gap > 10% — model may be overfitting")
elif acc[-1] < 0.70:
    print("  Train accuracy < 70% — model may be underfitting")
else:
    print(" Model is well-fitted!")

## Step 11: Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_gen, verbose=1)

print(f"\n{'='*40}")
print(f"  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"{'='*40}")

## Step 12: Predictions & Confusion Matrix

In [ ]:
# True labels (test_gen.classes works because shuffle=False)
y_true = test_gen.classes

# Predictions
predictions = model.predict(test_gen, verbose=1)
y_pred      = np.argmax(predictions, axis=1)

# Class names sorted by index
class_labels = [k for k, v in sorted(test_gen.class_indices.items(), key=lambda x: x[1])]

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels,
            yticklabels=class_labels,
            linewidths=0.5)
plt.title('Apple Disease Classification — Confusion Matrix', fontsize=14)
plt.ylabel('Actual Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 13: Classification Report

In [ ]:
print("\n--- Detailed Classification Report ---")
print(classification_report(y_true, y_pred, target_names=class_labels))

## Step 14: Save Best Model to Google Drive

In [ ]:
save_path = '/content/drive/MyDrive/apple_disease_cnn_best.keras'
model.save(save_path)
print(f" Model saved to: {save_path}")